## Visualizing Features with Galaxy10 DECals Dataset

The Galaxy10 DECals dataset consists of 17,736 256 $\times$ 256 images of galaxies classified into 10 categories. The underlying images were originally sourced from the _Sloan Digital Sky Survey (SDSS)_ via the Galaxy Zoo project. Here we train a model and visualize the learned feature in a number of different ways.

In [ ]:
import torch
import torchvision
import torchvision.transforms.functional as TF
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
from torchvision import models, transforms
from torch.autograd import Variable
import numpy as np
from tqdm.notebook import trange
import h5py
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from PIL import Image
from captum.attr import LayerGradCam, Saliency
from matplotlib import cm

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load the Data

In [ ]:
with h5py.File('Galaxy10_DECals.h5', 'r') as gF:
    images = np.array(gF['images'])
    labels = np.array(gF['ans'])

In [ ]:
from sklearn.utils import shuffle
images, labels = shuffle(images, labels)

### Prepare the Data

The data must be prepared in a similar fashion to the MNIST examples except now there are three channels instead of one. The classes must be converted to a categorical variable.

In [ ]:
train_idx, test_idx = train_test_split(np.arange(labels.shape[0]), test_size=0.1)
X_train, y_train, X_test, y_test = images[train_idx], labels[train_idx], images[test_idx], labels[test_idx]

encoder = OneHotEncoder(sparse_output=False)
train_y = encoder.fit_transform(y_train.reshape(-1, 1))
test_y = encoder.transform(y_test.reshape(-1, 1))

X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
X_train /= 255
X_test /= 255

X_train = torch.Tensor(X_train).permute(0, 3, 1, 2)
train_y = torch.Tensor(train_y)
train_dataset = TensorDataset(X_train, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, 
                                           shuffle=True, drop_last=True)

X_test = torch.Tensor(X_test).permute(0, 3, 1, 2)
test_y = torch.Tensor(test_y)
test_dataset = TensorDataset(X_test, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, 
                                          shuffle=False, drop_last=True)

### Build CNN Model

The CNN model is constructed using 3 Convolutional layers and 2 Max Pooling layers before finishing with a dense layer before the softmax. This is essentially the same structure as used with MNIST but the input size and number of channels has changed.

In [ ]:
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 32 * 32, 164)
        self.fc2 = nn.Linear(164, 10)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.softmax(self.fc2(x), dim=1)
        return x

In [ ]:
no_epochs = 200

In [ ]:
history_dict = dict()

### Train a model with Data Aumentation

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs, device):
    train_errors = []
    test_errors = []
    train_accuracies = []
    test_accuracies = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        correct_train = 0

        # Training
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)
            # Calculate accuracy
            predicted = torch.argmax(outputs.squeeze(), dim=1)
            targets = torch.argmax(batch_y, dim=1)
            correct_train += (predicted == targets).sum().item()

        train_loss /= len(train_loader.dataset)
        train_accuracy = 100 * correct_train / len(train_loader.dataset)
        train_errors.append(train_loss)
        train_accuracies.append(train_accuracy)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        correct_test = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)
                # Calculate accuracy
                predicted = torch.argmax(outputs.squeeze(), dim=1)
                targets = torch.argmax(batch_y, dim=1)
                correct_test += (predicted == targets).sum().item()

        test_loss /= len(test_loader.dataset)
        test_accuracy = 100 * correct_test / len(test_loader.dataset)
        test_errors.append(test_loss)
        test_accuracies.append(test_accuracy)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}, \
            Train Acc: {train_accuracy:.2f}%, Test Acc: {test_accuracy:.2f}%")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['train_acc'] = train_accuracies
    history['test_acc'] = test_accuracies
        
    return history

In [ ]:
class CustomAugmentations:
    def __init__(self, rotation_range, width_shift_range, height_shift_range, shear_range, zoom_range, horizontal_flip):
        self.rotation_range = rotation_range
        self.width_shift_range = width_shift_range
        self.height_shift_range = height_shift_range
        self.shear_range = shear_range
        self.zoom_range = zoom_range
        self.horizontal_flip = horizontal_flip

    def __call__(self, img):
        # Random rotation
        angle = torch.empty(1).uniform_(-self.rotation_range, self.rotation_range).item()
        img = TF.rotate(img, angle, interpolation=TF.InterpolationMode.NEAREST)

        # Random width shift
        width_shift = torch.empty(1).uniform_(-self.width_shift_range, self.width_shift_range).item()
        img = TF.affine(img, angle=0, translate=(width_shift * img.shape[2], 0), scale=1, shear=0, interpolation=TF.InterpolationMode.NEAREST)

        # Random height shift
        height_shift = torch.empty(1).uniform_(-self.height_shift_range, self.height_shift_range).item()
        img = TF.affine(img, angle=0, translate=(0, height_shift * img.shape[1]), scale=1, shear=0, interpolation=TF.InterpolationMode.NEAREST)

        # Random shear
        shear = torch.empty(1).uniform_(-self.shear_range, self.shear_range).item()
        img = TF.affine(img, angle=0, translate=(0, 0), scale=1, shear=shear, interpolation=TF.InterpolationMode.NEAREST)

        # Random zoom
        zoom = torch.empty(1).uniform_(1 - self.zoom_range, 1 + self.zoom_range).item()
        img = TF.affine(img, angle=0, translate=(0, 0), scale=zoom, shear=0, interpolation=TF.InterpolationMode.NEAREST)

        # Random horizontal flip
        if self.horizontal_flip and torch.rand(1).item() > 0.5:
            img = TF.hflip(img)
        
        return img

custom_transforms = transforms.Compose([
    CustomAugmentations(rotation_range=40, 
                        width_shift_range=0.2, 
                        height_shift_range=0.2, 
                        shear_range=0.2, 
                        zoom_range=0.2, 
                        horizontal_flip=True)
])

class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

aug_train_dataset = AugmentedDataset(train_dataset, 
                                     transform=custom_transforms)

train_loader = DataLoader(aug_train_dataset, batch_size=128, shuffle=True)

In [ ]:
cnn_model = CNNModel().to(device)

In [ ]:
# Define the optimizer (Adam in this case)
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

# Define the loss function (categorical cross-entropy in this case)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
history = train_model(cnn_model, train_loader, test_loader,
                      loss_fn, optimizer, no_epochs, device)
history_dict['CNN Visual'] = history

In [ ]:
torch.save(cnn_model.state_dict(), "cnn_visual.pth")

In [ ]:
cnn_model.load_state_dict(torch.load("cnn_visual.pth"))

### Visualizing Filters

The simplest type of visualization is to extract the filters from each layer and plot them.

In [ ]:
print(cnn_model)

In [ ]:
layer = cnn_model.conv1

Extract the weights and normalize them for visualization

In [ ]:
weights = layer.weight.data.cpu().numpy()

In [ ]:
max_val = weights.max()
min_val = weights.min()
abs_max = max(abs(min_val), abs(max_val))

In [ ]:
weights = (weights / abs_max) * 255

In [ ]:
weights = weights.astype(np.uint8)

In [ ]:
plt.figure(figsize=(15, 5))
n_filters, ix = 16, 1
for i in range(n_filters):
    f = weights[i]
    # Loop over the RGB channels
    for j in range(3):
        ax = plt.subplot(3, n_filters, ix)
        ax.set_xticks([])
        ax.set_yticks([])
        # Plot filter channel in grayscale
        plt.imshow(f[j], cmap='gray')
        ix += 1
plt.savefig('Filters.png', dpi=300, bbox_inches='tight')

### Visualizing Intermediate Activations

Another simple visualization technique is to view the activations in each channel and layer in response to an image. We load an image taken by the Hubble Space Telescope that is not part of the original training set.

In [ ]:
img_filename = 'potw2123a.jpg'
img = Image.open(img_filename).convert('RGB')
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])
img_tensor = transform(img).unsqueeze(0)

In [ ]:
# Visualize the image
plt.imshow(img_tensor[0].permute(1, 2, 0).cpu().detach().numpy())
plt.show()

In [ ]:
img_tensor = img_tensor.to(device)

First we predict what type of galaxy this is using the model

In [ ]:
cnn_model.eval()
with torch.no_grad():
    output = cnn_model(img_tensor)
ipred = torch.argmax(output, dim=1).item()

In [ ]:
gal10_classes = ['Disturbed', 'Merging', 'Round Smooth',\
                 'In-Between Round Smooth', 'Cigar-shaped Smooth', \
                 'Barred Spiral', 'Unbarred Tight Spiral', \
                 'Unbarred Loose Spiral', 'Edge-on without Bulge', \
                 'Edge-on with Bulge']
print(gal10_classes[ipred])

Extract the activations

In [ ]:
# Extract the activations
act = {}
def get_activation(name):
    def hook(model, input, output):
        act[name] = output.detach()
    return hook

for name, layer in cnn_model.named_modules():
    if isinstance(layer, torch.nn.Conv2d):
        layer.register_forward_hook(get_activation(name))

# Perform a forward pass to get the activations
with torch.no_grad():
    _ = cnn_model(img_tensor)

Define a function to plot all the intermediate activations for a given layer

In [ ]:
def plotActivations(layer_act, img_per_row, filename=None, cmap='gist_gray'):
    layer_act = layer_act.squeeze().cpu().numpy()
    features = layer_act.shape[0]
    size = layer_act.shape[1]
    cols = features // img_per_row
    display_grid = np.zeros((size * cols, img_per_row * size))
    
    for col in range(cols):
        for row in range(img_per_row):
            ch_img = layer_act[col * img_per_row + row]
            
            # Enhance the contrast
            ch_img_min = ch_img.min()
            ch_img_max = ch_img.max()
            diff = (ch_img_max - ch_img_min + 1e-5)
            ch_img = ch_img * 255 / diff
            display_grid[col * size : (col + 1) * size,
                         row * size : (row + 1) * size] = ch_img
                
    scale = 1.0 / size
    plt.figure(figsize=(scale * display_grid.shape[1],
                        scale * display_grid.shape[0]))
    plt.grid(False)
    
    fig = plt.imshow(display_grid, aspect='auto', cmap=cmap, vmin=0, vmax=255)
    fig.axes.get_xaxis().set_visible(False)
    fig.axes.get_yaxis().set_visible(False)
    if filename is not None:
        plt.savefig(filename, dpi=300, bbox_inches='tight')

In [ ]:
plotActivations(act['conv1'], 8, 'Conv2D_3_BW')

In [ ]:
plotActivations(act['conv2'], 8, 'Conv2D_4_BW')

In [ ]:
plotActivations(act['conv3'], 8, 'Conv2D_5_BW')

In [ ]:
plotActivations(act['conv1'], 8, 'Conv2D_3', 'inferno')

In [ ]:
plotActivations(act['conv2'], 8, 'Conv2D_4', 'inferno')

In [ ]:
plotActivations(act['conv3'], 8, 'Conv2D_5', 'inferno')

### Saliency Map

A saliency map gives the regions of the image that contribute most to the output, in this case the designation of the galaxy type.

In [ ]:
cnn_model.eval()

# Replace the model's last activation function to a linear one
class ReplaceToLinear(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.replace_last_layer()

    def replace_last_layer(self):
        if isinstance(self.model.fc2, nn.Linear):
            self.model.fc2 = nn.Linear(self.model.fc2.in_features, self.model.fc2.out_features)
        else:
            # Replace the last activation layer if it's not a linear layer
            for name, module in self.model.named_modules():
                if isinstance(module, nn.ReLU):  # assuming ReLU is the last activation function
                    setattr(self.model, name, nn.Identity())

    def forward(self, x):
        return self.model(x)

cnn_model_da = ReplaceToLinear(cnn_model)
cnn_model_da = cnn_model_da.to(device)

In [ ]:
# Define the score function
class CategoricalScore:
    def __init__(self, target_class):
        self.target_class = target_class

    def __call__(self, model_output):
        return model_output[:, self.target_class]

In [ ]:
# Get the predicted class
score = CategoricalScore(ipred)

In [ ]:
# Create a Saliency object
saliency = Saliency(cnn_model_da)

In [ ]:
# Compute saliency map
saliency_map = saliency.attribute(img_tensor, target=ipred)

In [ ]:
# Convert the saliency map to a NumPy array and visualize it
saliency_map = saliency_map.squeeze().permute(1, 2, 0).cpu().numpy()

In [ ]:
sa_max = saliency_map.max()
sa_min = saliency_map.min()
saliency_map = (saliency_map - sa_min) * 255 / (sa_max - sa_min)

In [ ]:
saliency_map = saliency_map.astype(np.uint8)[:,:,0]

In [ ]:
fig = plt.imshow(saliency_map, cmap='binary')
fig.axes.get_xaxis().set_visible(False)
fig.axes.get_yaxis().set_visible(False)
plt.savefig('SaliencyMap.png', dpi=300, bbox_inches='tight')

### GradCAM

In [ ]:
layer_gradcam = LayerGradCam(cnn_model_da, cnn_model_da.model.conv3)

# Generate heatmap with GradCAM
attributions = layer_gradcam.attribute(img_tensor, target=ipred)

# Upsample the attributions to the input image size
upsampled_attributions = LayerGradCam.interpolate(attributions, img_tensor.shape[-2:])

# Convert the attributions to a NumPy array and visualize it
heatmap = upsampled_attributions.squeeze().cpu().detach().numpy()
heatmap = np.uint8(cm.jet(heatmap)[..., :3] * 255)

# Overlay the heatmap on the input image
img_array_np = img_tensor.squeeze().permute(1, 2, 0).cpu().detach().numpy()
img_array_np = np.uint8((img_array_np - img_array_np.min()) / (img_array_np.max() - img_array_np.min()) * 255)

fig, ax = plt.subplots()
ax.imshow(img_array_np)
ax.imshow(heatmap, cmap='jet', alpha=0.5)  # overlay
ax.axis('off')
plt.savefig('GradCAM.png', dpi=300, bbox_inches='tight')